In [17]:
import requests
import pandas as pd
from minio import Minio
from io import BytesIO
import datetime as dt
import os

In [19]:
# ========== КОНФИГУРАЦИЯ ==========
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "mjyQgjfKGdn0urK0V1vm"
MINIO_SECRET_KEY = "At6cf5sGWjAGnHgUzV18mDxXulvVp9MZ03v2FxuK"
MINIO_BUCKET = "earthquake-data"
USE_SECURE = False

# API параметры
API_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
LAYER = "raw"
SOURCE = "earthquake"
START_DATE = "2025-01-01"
END_DATE = "2025-01-31"

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=USE_SECURE
)

In [20]:
def get_date(start_date: str = None, end_date: str = None, day: int = 1) -> tuple[str,str]:
    if start_date:
        date_temp = (dt
                .datetime
                .strftime(
                    (
                        dt.datetime.strptime(start_date,'%Y-%m-%d')+dt.timedelta(days=day)
                    ),"%Y-%m-%d"
                )
            )
        start_date = date_temp
        
    if end_date:
        date_temp = (dt
                .datetime
                .strftime(
                    (
                        dt.datetime.strptime(end_date,'%Y-%m-%d')+dt.timedelta(days=day)
                    ),"%Y-%m-%d"
                )
            )
        end_date = date_temp

    if start_date and end_date:
        return start_date, end_date
        
    return date_temp
    

In [12]:
def get_df_from_api(start_date: str, end_date: str) -> pd.DataFrame:
    all_data = []
    
    params = {
        "format": "csv",
        "starttime": start_date,
        "endtime": end_date,
    }
    
    response = requests.get(API_URL, params=params)
    
    if response.status_code != 200:
        return pd.DataFrame()
    
    all_data = []
    
    content = response.text
    lines = content.strip().split('\n')
        
    data_lines = lines[1:]  # Пропускаем заголовок
    all_data.extend(data_lines)
        
    # Преобразуем в DataFrame
    if all_data:
        headers = lines[0].split(',')
        data_rows = [line.replace(', ',';').split(',') for line in all_data]
        df = pd.DataFrame(data_rows, columns=headers)
        df['place'] = df['place'].str.split(';').str.join(', ')

    return df

In [15]:
start_date = START_DATE
end_date = get_date(end_date=START_DATE)

while start_date != END_DATE:
    df = get_df_from_api(start_date=start_date, end_date=end_date)
    if not df.empty:
        parquet_buffer = BytesIO()
        df.to_parquet(parquet_buffer, index=False)
        parquet_buffer.seek(0)
        
        parquet_path = f"{LAYER}/{SOURCE}/{start_date}/{start_date}_00-00-00.gz.parquet"
        client.put_object(
            MINIO_BUCKET,
            parquet_path,
            parquet_buffer,
            length=len(parquet_buffer.getvalue()),
            content_type="application/parquet"
        )
        
    start_date, end_date = get_date(start_date=start_date, end_date=end_date)